# 04 · Thinking off (stage S2)

The chat template ignores `add_generation_prompt` and `reasoning_effort="none"` does not stop thinking, so the start of the
model reply is appended to the prompt (`prefill`). Variants:

- **A** `text`: opens the answer message.
- **B** `empty_think`: an empty thinking message, then the answer message.

Done-when: the better variant parses at least 95% on 20 GQA and 20 TextVQA dev items. Then pick `res_px` (768 vs 1024).


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
# GPU choice must precede CUDA init, which importing adaloop.api already does: the one import outside adaloop.api
from utils.gpus import select_gpu_group

CONFIG = "configs/mvp.yaml"   # the config this notebook passes to setup_notebook
GPU_GROUP = None  # None = first free pair of gpu_groups; 0 or 1 forces a pair; changing it needs a kernel restart
GPU_IDS = select_gpu_group(CONFIG, GPU_GROUP)

In [ ]:
import math
import os
import re
import socket


In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook("configs/mvp.yaml")
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
# torch is imported here because the GPU list needs it; the other third-party imports come after this cell
import torch

IS_INARA = "inara" in socket.gethostname()
print("host", socket.gethostname(), "| on Inara:", IS_INARA)
print("chosen GPUs, physical ids CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"chosen GPU {i} (torch index) | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")
else:
    print("no CUDA GPU visible: the model steps will fail, the rest of the notebook can still be read")


In [ ]:
import pandas as pd


In [ ]:
# Settings: batch sizes and the loop count used for every evaluation in this notebook
N_ITEMS = 20  # items per benchmark in step 4
N_RES = 50  # TextVQA items in step 5
R = 3  # loop passes for every evaluation here
MAX_NEW = 16  # short cap: enough for a one-word answer
RES = [768, 1024]  # long-side sizes compared in step 5
PREFILLS = {
    "A": "<|message_model|><|content_text|>",
    "B": "<|message_model|><|content_thinking|><|end_message|><|message_model|><|content_text|>",
}

# Expensive batch runs. Set False to skip them; the cells that need their output then skip too.
RUN_VARIANTS = True  # step 4: N_ITEMS items x 2 benchmarks x 2 variants
RUN_RES_PX = True  # step 5: N_RES TextVQA items x 2 sizes
# The variant used in steps 5 and 6. Step 4 fills it in; set "A" or "B" by hand when RUN_VARIANTS is False.
PREFILL = None


### Step-1:- Load the model

The model and its processor are loaded once and reused by every step below. Loading is the slow part of the notebook.


In [ ]:
print("[STAGE] load model")
vlm = api.load_model(cfg)
tok = vlm.processor.tokenizer


### Step-2:- Chat template and special tokens

The prefill is made of special-token markers, so first look at what the template produces and which ids those markers have.


In [ ]:
print(vlm.processor.chat_template or tok.chat_template)


In [ ]:
for token, token_id in sorted(tok.get_added_vocab().items(), key=lambda kv: kv[1]):
    if "unused" not in token:  # skip the reserved placeholder tokens
        print(token_id, token)
print("eos", tok.eos_token_id, tok.eos_token)


In [ ]:
# reasoning_effort="none" is passed here, yet the template still leaves the reply open for thinking
question = [{"role": "user", "content": [{"type": "text", "text": "What color is the sky?"}]}]
print(repr(vlm.processor.apply_chat_template(question, reasoning_effort="none", add_generation_prompt=True, tokenize=False)))


In [ ]:
# Each marker must be exactly one token id, or the prefill would not be what the model itself emits
for name, prefill in PREFILLS.items():
    ids = tok(prefill, add_special_tokens=False)["input_ids"]
    tokens = tok.convert_ids_to_tokens(ids)
    print(name, ids, tokens)
    decoded = tok.decode(ids, skip_special_tokens=False)
    assert decoded == prefill, f"prefill {name} does not decode back: {decoded!r}"
    assert tokens == re.findall(r"<\|[^|]+\|>", prefill), f"prefill {name} markers are not one id each: {tokens}"
print("✓ template: both prefills decode back and are one id per marker")


### Step-3:- One item: baseline vs prefill

This is the sample run before any batch: one GQA item, sent three ways. Same image and question, so only the prefill differs.


In [ ]:
print("[STAGE] one item inputs")
item = api.load_gqa(cfg, "dev", limit=1, root=ROOT)[0]
prompt = api.item_prompt(item, cfg)
image = api.resize_long_side(item.image, cfg["res_px"])
print(repr(prompt), "->", item.answers)

base = api.prepare_inputs(vlm, prompt, image)
n = base["input_ids"].shape[-1]  # length of the prompt without any prefill
inputs = {"baseline": base}
for name, prefill in PREFILLS.items():
    inputs[name] = api.prepare_inputs(vlm, prompt, image, prefill=prefill)


In [ ]:
# A prefill must only append ids after the prompt; the prompt itself and the image tokens must not change
for name in PREFILLS:
    ids = inputs[name]["input_ids"][0].tolist()
    mask = inputs[name]["attention_mask"]
    assert ids[:n] == base["input_ids"][0].tolist(), f"{name}: prompt ids changed"
    assert ids[n:] == tok(PREFILLS[name], add_special_tokens=False)["input_ids"], f"{name}: tail is {ids[n:]}"
    assert mask.shape == inputs[name]["input_ids"].shape and bool(mask.all()), f"{name}: attention mask is not all ones"
print("✓ one item inputs: prefill only appends ids")


In [ ]:
print("[STAGE] one item generate")
api.set_loops(vlm, R)
for name, inp in inputs.items():
    # the baseline keeps the long limit so its thinking is visible
    max_new = cfg["max_new_tokens"] if name == "baseline" else MAX_NEW
    gen = api.generate(vlm, inp, max_new)
    print(f"--- {name}: {len(gen.token_ids)} tokens, hit_limit={gen.hit_limit}")
    print(repr(gen.raw_text))
    # A and B open the answer message, so their output starts at the answer; the baseline does not
    print("extracted:", repr(api.extract_answer(gen.raw_text, gen.hit_limit, prefilled=name in PREFILLS)))


In [ ]:
# With a prefill the generated text starts at the answer itself (no <|content_text|> marker): prefilled=True
SOAP = "sink<|end_message|><|message_model|><|content_text|>The user is asking what the soap dispenser is sitting on"
cases = [  # (raw output, hit_limit, prefilled, expected answer)
    ("blue<|end_message|><|content_model_end_sampling|>", False, True, "blue"),
    ("blue<|end_message|>", True, True, "blue"),
    ("a long answer that got cut", True, True, None),  # cut before <|end_message|>: unparseable
    (SOAP, True, True, "sink"),  # a second message after the answer is ignored
    ("<|end_message|>", False, True, None),  # empty answer
    # without a prefill the answer follows the last <|content_text|> marker
    ("<|content_thinking|>hm<|end_message|><|message_model|><|content_text|>blue<|end_message|>", False, False, "blue"),
]
for raw, hit, prefilled, want in cases:
    got = api.extract_answer(raw, hit, prefilled=prefilled)
    assert got == want, f"extract_answer({raw[:20]!r}, hit_limit={hit}, prefilled={prefilled}) gave {got!r}, want {want!r}"
print("✓ one item: extraction on fixed outputs")

### Step-4:- Parse rate of A and B (done-when)

The batch run: N_ITEMS GQA and N_ITEMS TextVQA dev items per variant. The worst benchmark of each variant decides the
winner, and the done-when is that the winner parses at least 95% on both. `accuracy` and `parse_rate` are percentages.


In [ ]:
# One evaluation with an in-memory config copy; the file name ends with the hash of that copy
def run(bench, bench_items, variant, res_px):
    """One evaluation of `bench_items` with `variant` at `res_px`; return (summary, rows)."""
    run_cfg = {**cfg, "answer_prefill": PREFILLS[variant], "max_new_tokens": MAX_NEW, "res_px": res_px}
    name = f"{bench}_dev_n{len(bench_items)}_res{res_px}_R{R}_{variant}_{api.config_hash(run_cfg)}.jsonl"
    out = ROOT / cfg["out_dir"] / "nb04" / name
    summary = api.run_evaluation(vlm, bench_items, run_cfg, R, out, resume=True)
    return summary, api.read_results(out)


In [ ]:
if RUN_VARIANTS:
    print("[STAGE] variants")
    items = {
        "gqa": api.load_gqa(cfg, "dev", limit=N_ITEMS, root=ROOT),
        "textvqa": api.load_textvqa(cfg, "dev", limit=N_ITEMS, root=ROOT),
    }
    table, variant_rows = [], {}
    for variant in PREFILLS:
        for bench, bench_items in items.items():
            summary, rows = run(bench, bench_items, variant, cfg["res_px"])
            variant_rows[variant, bench] = rows
            table.append({"variant": variant, "benchmark": bench, "n": summary["n"], "accuracy": summary["accuracy"],
                          "parse_rate": summary["parse_rate"], "n_out_tokens_mean": summary["n_out_tokens_mean"]})
    table = pd.DataFrame(table)
table if RUN_VARIANTS else "skipped: RUN_VARIANTS is False"


In [ ]:
if RUN_VARIANTS:
    for (variant, bench), rows in variant_rows.items():
        print(f"--- {variant} {bench}")
        for r in rows[:3]:
            print(f"  {r['raw_output']!r} -> {r['pred']!r}  answers {r['answers'][:3]}")
        for r in [r for r in rows if not r["parse_ok"]][:3]:
            print(f"  UNPARSED {r['raw_output']!r}")
else:
    print("skipped: RUN_VARIANTS is False")


In [ ]:
# Pick the variant whose worst benchmark parse rate is higher
if RUN_VARIANTS:
    worst = table.groupby("variant").parse_rate.min()
    best = "A" if worst["A"] >= worst["B"] else "B"
    PREFILL = best
    print("lowest parse rate per variant", worst.to_dict(), "| best", best, "| PREFILL", PREFILL)
else:
    assert PREFILL in PREFILLS, "RUN_VARIANTS is False: set PREFILL to 'A' or 'B' in the settings cell"
    print("skipped: RUN_VARIANTS is False; using PREFILL", PREFILL, "from the settings")


In [ ]:
if RUN_VARIANTS:
    assert all(len(rows) == N_ITEMS for rows in variant_rows.values()), [len(r) for r in variant_rows.values()]
    assert worst[PREFILL] >= 95.0, f"{PREFILL}: lowest parse rate {worst[PREFILL]:.1f} is below 95"
    print("✓ variants: the better variant parses >= 95% on both benchmarks")
else:
    print("skipped: RUN_VARIANTS is False")


### Step-5:- res_px: 768 vs 1024 on TextVQA dev

`resize_long_side` never upscales, so an image whose long side is already below `res_px` is sent as it is.
The processor makes one visual token per patch, so the expected token count below is a check on the resize.


In [ ]:
res_items = api.load_textvqa(cfg, "dev", limit=N_RES, root=ROOT)
long_sides = [max(it.image.size) for it in res_items]
print("original long side: min", min(long_sides), "max", max(long_sides))
patch = vlm.processor.image_processor.size
print("patch", patch)

expected = {}
for res_px in RES:
    sizes = [api.resize_long_side(it.image, res_px).size for it in res_items]
    expected[res_px] = sum(math.ceil(w / patch["width"]) * math.ceil(h / patch["height"]) for w, h in sizes) / len(sizes)
    print(f"res_px {res_px}: {sum(s < res_px for s in long_sides)}/{len(res_items)} images smaller than res_px, "
          f"largest sent {max(max(s) for s in sizes)}, expected mean n_vis {expected[res_px]:.1f}")


In [ ]:
if RUN_RES_PX:
    assert PREFILL in PREFILLS, "set PREFILL to 'A' or 'B' (step 4 does this, or set it in the settings)"
    print("[STAGE] res_px")
    res_table = []
    for res_px in RES:
        summary, rows = run("textvqa", res_items, PREFILL, res_px)
        res_table.append({"res_px": res_px, "variant": PREFILL, "n": len(rows), "accuracy": summary["accuracy"],
                          "parse_rate": summary["parse_rate"], "n_vis_mean": summary["n_vis_mean"],
                          "n_vis_expected": expected[res_px], "generate_s_mean": summary["generate_s_mean"],
                          "peak_mem_gb_max": summary["peak_mem_gb_max"]})
    res_table = pd.DataFrame(res_table)
res_table if RUN_RES_PX else "skipped: RUN_RES_PX is False"


In [ ]:
if RUN_RES_PX:
    assert (res_table.n == N_RES).all(), res_table.n.tolist()
    print("✓ res_px: every size has all N_RES rows")
else:
    print("skipped: RUN_RES_PX is False")


### Step-6:- Decision to log

Paste these lines into `docs/decisions.md`, fill in the blanks, then set `answer_prefill`, `res_px` and `max_new_tokens` in `configs/mvp.yaml`.


In [ ]:
if RUN_VARIANTS and RUN_RES_PX:
    parse = {bench: value for bench, value in table[table.variant == PREFILL].set_index("benchmark").parse_rate.items()}
    print(f"## <date> · S2 · thinking off and res_px (git {api.git_hash()})")
    print(f"- Decision: answer_prefill = variant {PREFILL} {PREFILLS[PREFILL]!r}, max_new_tokens = {MAX_NEW}; "
          f"parse rate GQA {parse['gqa']:.1f}, TextVQA {parse['textvqa']:.1f} (n={N_ITEMS} each, R={R}, res_px={cfg['res_px']}).")
    summary_text = "; ".join(f"{r.res_px}: acc {r.accuracy:.1f}, n_vis {r.n_vis_mean:.0f}, {r.generate_s_mean:.2f}s, "
                             f"{r.peak_mem_gb_max:.1f} GB" for r in res_table.itertuples())
    print(f"- Decision: res_px = <768|1024> because <reason>. TextVQA dev n={N_RES}: {summary_text}.")
else:
    print("skipped: this decision needs RUN_VARIANTS and RUN_RES_PX both True")


### Step-7:- Figure F02

Thinking-off comparison: (a) parse rate and (b) accuracy of prefill variants A and B on GQA and TextVQA, and (c) TextVQA accuracy at 768 vs 1024 px. It shows the evidence behind the prefill and `res_px` decisions in step 6.

In [ ]:
# F02 needs the step 4 table and the step 5 table, so it is built only when both ran
if RUN_VARIANTS and RUN_RES_PX:
    api.set_paper_style()
    fig = api.plot_thinking_off(table, res_table)
else:
    print("skipped: RUN_VARIANTS and RUN_RES_PX must both be True to plot F02")

In [ ]:
if RUN_VARIANTS and RUN_RES_PX:
    pdf_path, png_path = api.save_figure(fig, "F02_thinking_off", final=False)
else:
    print("skipped: F02 not saved: RUN_VARIANTS and RUN_RES_PX must both be True")

In [ ]:
if RUN_VARIANTS and RUN_RES_PX:
    assert pdf_path.is_file() and png_path.is_file(), f"missing {pdf_path} or {png_path}"
    print("✓ F02_thinking_off draft saved as PDF and PNG (final=False, paper/ untouched)")
else:
    print("skipped: F02 not saved: RUN_VARIANTS and RUN_RES_PX must both be True")

In [ ]:
print("✓ notebook 04 done | PREFILL", PREFILL)
